In [10]:
import requests
from bs4 import BeautifulSoup
import time
import random
from urllib.parse import urljoin, quote_plus
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
from dotenv import load_dotenv

load_dotenv()

URL_BASE = "https://books.toscrape.com/"
API_KEY = os.getenv("GOOGLE_API_KEY")

if not API_KEY:
    print("⚠️ ADVERTENCIA: No se encontró la API KEY. Configura tu archivo .env")

ESTRELLAS = {
    'One': 1, 
    'Two': 2, 
    'Three': 3, 
    'Four': 4, 
    'Five': 5
}

def crear_sesion():
    session = requests.Session()
    session.headers.update({'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36'})
    return session

def obtener_soup(url, session):
    try:
        respuesta = session.get(url)    
        respuesta.raise_for_status()
        soup = BeautifulSoup(respuesta.text, 'lxml')
        return soup
    
    except Exception as error:
        print(f"Error en {url}: {error}")
        return None
    
def extraer_datos_libro(libros, categoria_nombre):
    titulo = libros.find('h3').a['title']
    precio_text = libros.find("p", class_="price_color").text
    precio_simb = precio_text.replace("£", "").replace('Â', '').strip()
    precio = float(precio_simb)
    rating = libros.find("p", class_="star-rating").get("class")
    stock = libros.find("p", class_="instock availability").get_text(strip=True)

    return {
        "titulo": titulo,
        "autor": "Pendiente",
        "precio": precio,
        "categoria": categoria_nombre,
        "rating": rating[1],
        "stock": stock
    }

def obtener_links_categorias(session):
    soup = obtener_soup(URL_BASE, session)
    lista_categoria = []

    panel_lateral = soup.find("div", class_="side_categories")
    enlaces = panel_lateral.find_all("a")

    for enlace in enlaces[1:]:
        nombre_cat = enlace.text.strip()
        ruta_relativa = enlace["href"]
        link_completo = URL_BASE + ruta_relativa

        lista_categoria.append({
            "nombre": nombre_cat,
            "url": link_completo
        })

    return lista_categoria

def obtener_autor(libro):
    titulo = libro['titulo']
    
    if API_KEY and random.random() < 0.30: # 30% de las veces consultamos
        try:
            # Construimos la URL segura
            titulo_codificado = quote_plus(titulo)
            url = f"https://www.googleapis.com/books/v1/volumes?q=intitle:{titulo_codificado}&key={API_KEY}&fields=items(volumeInfo/authors)"
            
            resp = requests.get(url, timeout=3)
            
            if resp.status_code == 200:
                data = resp.json()
                # Google devuelve una lista 'items', verificamos si existe
                if "items" in data:
                    info = data["items"][0]["volumeInfo"]
                    # Verificamos si tiene autores
                    if "authors" in info:
                        autor_real = info["authors"][0] # Tomamos el primero
                        libro['autor'] = autor_real
                        return f"✅ Google: {titulo[:15]}... -> {autor_real}"
            
        except Exception as e:
            pass # Si falla, usamos el simulado
            
        # Google permite más peticiones, pero mantenemos una pausa corta
        time.sleep(0.5)
    
    # Fallback: Si no hay key, falló la búsqueda o cayó en el 70% restante
    libro['autor'] = "Autor Simulado"
    return f"🤖 Simulado: {titulo[:15]}..."

def procesar_categoria(cat, session):
    """Función que será ejecutada por cada hilo para procesar una categoría completa"""
    nombre = cat["nombre"]
    url_actual = cat["url"]
    libros_de_esta_cat = []
    
    while True:
        soup = obtener_soup(url_actual, session)
        if not soup:
            break

        articulos = soup.find_all("article", class_="product_pod")
        for art in articulos:
            datos = extraer_datos_libro(art, nombre)
            libros_de_esta_cat.append(datos)

        boton_next = soup.find("li", class_="next")
        if boton_next:
            enlace_next = boton_next.a['href']
            url_actual = urljoin(url_actual, enlace_next)
        else:
            break
            
    # Imprimimos el resultado al finalizar la categoría
    print(f"✅ Categoría procesada: {nombre:.<30} {len(libros_de_esta_cat)} libros")
    return libros_de_esta_cat

def ejecutar_scraping():
    # --- FASE 1: SCRAPING DEL SITIO (Descarga masiva) ---
    print("🚀 FASE 1: Descargando libros de todas las categorías...")
    inicio = time.time()
    
    mi_sesion = crear_sesion()
    categorias = obtener_links_categorias(mi_sesion)
    
    lista_libros = []
    
    # 5 hilos para descargar las categorías rápido
    with ThreadPoolExecutor(max_workers=5) as executor:
        # Mapeamos la función procesar_categoria a cada categoría encontrada
        resultados = list(executor.map(lambda c: procesar_categoria(c, mi_sesion), categorias))
        
    # Unimos los resultados de todos los hilos en una sola lista
    for sublista in resultados:
        lista_libros.extend(sublista)
        
    tiempo_fase1 = time.time() - inicio
    print(f"\n📦 FASE 1 COMPLETADA: {len(lista_libros)} libros descargados en {tiempo_fase1:.2f}s.")
    
    # --- FASE 2: ACTUALIZACIÓN DE AUTORES (Aquí se quita el "Pendiente") ---
    print("\n🕵️ FASE 2: Buscando autores para cada libro...")
    
    # Usamos ThreadPoolExecutor para actualizar los autores en paralelo
    # max_workers=4 es seguro para la API
    with ThreadPoolExecutor(max_workers=4) as executor:
        # Enviamos cada libro a la función que busca el autor
        # IMPORTANTE: Los diccionarios se modifican 'in-place' (se actualizan solos)
        futures = [executor.submit(obtener_autor, libro) for libro in lista_libros]
        
        # Esperamos a que terminen todos para asegurar que no quede ningún "Pendiente"
        total = len(futures)
        for i, future in enumerate(as_completed(futures), 1):
            if i % 50 == 0: # Imprimimos progreso cada 50 libros
                print(f"   ...Progreso Autores: {i}/{total} completados")

    print(f"\n🎉 ¡Misión Cumplida! Todos los datos están listos.")
    return lista_libros

if __name__ == "__main__":
    lista_libros = ejecutar_scraping()
    print("\nDatos Libros")
    for i, libro in enumerate(lista_libros, 1):
        titulo = libro.get('titulo', 'N/A')
        autor = libro.get('autor', 'N/A')
        precio = libro.get('precio', 0.0)
        categoria = libro.get('categoria', 'N/A')
        rating = libro.get('rating', 0)
        stock = libro.get('stock', 'N/A')

        print(f"{i}- | {titulo} | {autor} | £{precio} | {categoria} | ⭐ {rating} | {stock}")

🚀 FASE 1: Descargando libros de todas las categorías...
✅ Categoría procesada: Historical Fiction............ 26 libros
✅ Categoría procesada: Philosophy.................... 11 libros
✅ Categoría procesada: Romance....................... 35 libros
✅ Categoría procesada: Womens Fiction................ 17 libros
✅ Categoría procesada: Travel........................ 11 libros
✅ Categoría procesada: Classics...................... 19 libros
✅ Categoría procesada: Religion...................... 7 libros
✅ Categoría procesada: Mystery....................... 32 libros
✅ Categoría procesada: Childrens..................... 29 libros
✅ Categoría procesada: Music......................... 13 libros
✅ Categoría procesada: Sequential Art................ 75 libros
✅ Categoría procesada: Fiction....................... 65 libros
✅ Categoría procesada: Science Fiction............... 16 libros
✅ Categoría procesada: Sports and Games.............. 5 libros
✅ Categoría procesada: New Adult..................